In [82]:
import pandas as pd
import numpy as np
import sys
import os
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [83]:
sys.path.append("/home/coralie/workspace/ProjectII/solubility_prediction_nmr/src/controller")
sys.path.append("/home/coralie/workspace/ProjectII/solubility_prediction_nmr/src")

from predictor_runner import generate_ml_ready_file_hansen

ici il va chercher le dossier generate_ml_ready_file_onehot_solvent dans predictor_runner

In [84]:
solvent_dict = {
    'MeOH': {'Hansen_D': 15.1, 'Hansen_P': 12.3, 'Hansen_H': 22.3},
    'ACN': {'Hansen_D': 15.3, 'Hansen_P': 18.0, 'Hansen_H': 6.1},
    'DMSO': {'Hansen_D': 18.4, 'Hansen_P': 16.4, 'Hansen_H': 10.2},
    'DCM': {'Hansen_D': 18.2, 'Hansen_P': 6.3, 'Hansen_H': 6.1},
    'CHCl3': {'Hansen_D': 17.8, 'Hansen_P': 3.1, 'Hansen_H': 5.7}
}

print(f"Solvants configurés : {list(solvent_dict.keys())}")
for name, params in solvent_dict.items():
    print(f"  {name} : {params}")

Solvants configurés : ['MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3']
  MeOH : {'Hansen_D': 15.1, 'Hansen_P': 12.3, 'Hansen_H': 22.3}
  ACN : {'Hansen_D': 15.3, 'Hansen_P': 18.0, 'Hansen_H': 6.1}
  DMSO : {'Hansen_D': 18.4, 'Hansen_P': 16.4, 'Hansen_H': 10.2}
  DCM : {'Hansen_D': 18.2, 'Hansen_P': 6.3, 'Hansen_H': 6.1}
  CHCl3 : {'Hansen_D': 17.8, 'Hansen_P': 3.1, 'Hansen_H': 5.7}


même chose que dans l'entrainement. 

In [85]:
matrix_ids_path = os.path.expanduser("~/workspace/ProjectII/solubility_prediction_nmr/data/all_matrix_ids.joblib")
all_matrix_ids = joblib.load(matrix_ids_path)

print(f"Matrix IDs chargées : {len(all_matrix_ids)}")

Matrix IDs chargées : 1


In [86]:
test_path = os.path.expanduser("~/workspace/ProjectII/solubility_prediction_nmr/data/test.csv")
df_test = pd.read_csv(test_path)
df_test.columns = df_test.columns.str.strip()

print(f"Fichier test chargé : {test_path}")
print(f"Nombre de lignes : {len(df_test)}")

model_path = os.path.expanduser("~/workspace/ProjectII/solubility_prediction_nmr/data/xgboost_all_solvents_model.joblib")
model = joblib.load(model_path)

print(f"Modèle chargé : {model_path}")

Fichier test chargé : /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/test.csv
Nombre de lignes : 22
Modèle chargé : /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/xgboost_all_solvents_model.joblib


Ici il charge ducoup le modèle avec 15% des molécules mises de coté. Il va ensuite charger notre modèle XGboost pour pouvoir s'en servir pour tester. 

In [87]:
def build_all_solvents_df(cleaned_df, solvent_dict, all_matrix_ids):
    all_dfs = []
    for solvent_name in solvent_dict.keys():
        if solvent_name not in cleaned_df.columns:
            print(f"Colonne '{solvent_name}' absente, ignorée.")
            continue
        print(f"Traitement du solvant : {solvent_name}...")
        solvent_ml_df = generate_ml_ready_file_hansen(
            cleaned_df, solvent_name, solvent_dict, all_matrix_ids
        )
        solvent_ml_df['Solvent_Name'] = solvent_name   # ← cette ligne est-elle bien là ?
        print(f"  → {len(solvent_ml_df)} lignes générées pour {solvent_name}")
        all_dfs.append(solvent_ml_df)
    return pd.concat(all_dfs, ignore_index=True)

Même truc ou on empile les solvants, sauf qu'on le fait sur le document de test. .head() permet d'afficher les premières lignes d'un Dataframe avec Pandas. Sans argument ça montre les 5 premières lignes.  

In [88]:
import inspect
print(inspect.getsource(build_all_solvents_df))

def build_all_solvents_df(cleaned_df, solvent_dict, all_matrix_ids):
    all_dfs = []
    for solvent_name in solvent_dict.keys():
        if solvent_name not in cleaned_df.columns:
            print(f"Colonne '{solvent_name}' absente, ignorée.")
            continue
        print(f"Traitement du solvant : {solvent_name}...")
        solvent_ml_df = generate_ml_ready_file_hansen(
            cleaned_df, solvent_name, solvent_dict, all_matrix_ids
        )
        solvent_ml_df['Solvent_Name'] = solvent_name   # ← cette ligne est-elle bien là ?
        print(f"  → {len(solvent_ml_df)} lignes générées pour {solvent_name}")
        all_dfs.append(solvent_ml_df)
    return pd.concat(all_dfs, ignore_index=True)



In [89]:
print("Construction du dataset de test (tous solvants empilés)...")
test_ml_df = build_all_solvents_df(df_test, solvent_dict, all_matrix_ids)
print(f"\nTerminé : {len(test_ml_df)} lignes au total (molécules × solvants)")

X_test_num = test_ml_df[['MolWt', 'RT']].values
X_test_matrix = np.vstack(test_ml_df['Matrix_Onehot'].values)
X_test_hansen = np.vstack(test_ml_df['Solvent_Hansen'].values)
X_test = np.hstack((X_test_num, X_test_matrix, X_test_hansen))
y_test = test_ml_df['Soluble'].values

print(f"X_test shape : {X_test.shape}")
print(f"y_test shape : {y_test.shape}")
print(test_ml_df.columns.tolist())

Construction du dataset de test (tous solvants empilés)...
Traitement du solvant : MeOH...
  → 22 lignes générées pour MeOH
Traitement du solvant : ACN...
  → 22 lignes générées pour ACN
Traitement du solvant : DMSO...
  → 22 lignes générées pour DMSO
Traitement du solvant : DCM...
  → 22 lignes générées pour DCM
Traitement du solvant : CHCl3...
  → 22 lignes générées pour CHCl3

Terminé : 110 lignes au total (molécules × solvants)
X_test shape : (110, 6)
y_test shape : (110,)
['MolWt', 'RT', 'Matrix_Onehot', 'Solvent_Hansen', 'Soluble', 'SMILES', 'Solvent_Name']


vstack empile les one-hot, hstack les colle. y_test corrigera pour savoir si le modèle à eu juste ou non. X_test et X_train a 1029 colonnes. pas compris ce 1029 colonnes. One-hot on cré eune liste de 0-4 pour 5 solvents. Chaque solvent est associé à un indice si par exemple le ACN est en position 1 ça créera un vecteur [01000]

In [90]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")

Accuracy  : 0.7818
Precision : 0.7917
Recall    : 0.8636


model.predict permet de demander au modèle de prédier pour chaque lignes si le composé est soluble ou non. accuracy donne la prop. de prédiction correct. precision, mesure les faux positif. recall mesure les faux négatifs. zero_division pour au cas ou le résultats doit être divisé par zéro. 

In [91]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

results_per_solvent = []

for solvent_name in solvent_dict.keys():
    mask = (test_ml_df['Solvent_Name'] == solvent_name).values
    if mask.sum() == 0:
        continue
    
    y_test_solvent = y_test[mask]
    y_pred_solvent = y_pred[mask]
    
    acc_s = accuracy_score(y_test_solvent, y_pred_solvent)
    prec_s = precision_score(y_test_solvent, y_pred_solvent, zero_division=0)
    rec_s = recall_score(y_test_solvent, y_pred_solvent, zero_division=0)
    
    results_per_solvent.append({
        'Solvent': solvent_name,
        'N': mask.sum(),
        'Accuracy': round(acc_s, 4),
        'Precision': round(prec_s, 4),
        'Recall': round(rec_s, 4)
    })

results_df = pd.DataFrame(results_per_solvent)
print(results_df)

  Solvent   N  Accuracy  Precision  Recall
0    MeOH  22    0.8636     1.0000  0.7857
1     ACN  22    0.8636     0.9091  0.8333
2    DMSO  22    0.7727     0.8333  0.8824
3     DCM  22    0.7273     0.7333  0.8462
4   CHCl3  22    0.6818     0.5882  1.0000
